# NB55: Spark + Redis Enrichment

Low-latency stream enrichment using Redis lookups.

## 1. Environment Setup

This cell installs **Java 8**, **Spark 3.5.0**, **Kafka 3.6.1**, and necessary Python libraries (`pyspark`, `kafka-python`, `redis`, `pymongo`, `elasticsearch`, `cassandra-driver`, `minio`). It also sets environment variables for Java and Spark.

In [ ]:
# Install Dependencies
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz
!tar xf spark-3.5.0-bin-hadoop3.tgz
!wget -q https://archive.apache.org/dist/kafka/3.6.1/kafka_2.13-3.6.1.tgz
!tar xf kafka_2.13-3.6.1.tgz
!pip install -q findspark pyspark kafka-python redis pymongo elasticsearch==7.10.1 cassandra-driver minio "numpy<2.0.0"

# Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.0-bin-hadoop3"
import findspark
findspark.init()

## 2. Start Services

This cell starts the required distributed services in the background:
- **Kafka & Zookeeper**: Event streaming platform.
- **Redis**: In-memory data store.

In [ ]:
# Start Kafka
!./kafka_2.13-3.6.1/bin/zookeeper-server-start.sh -daemon ./kafka_2.13-3.6.1/config/zookeeper.properties
!./kafka_2.13-3.6.1/bin/kafka-server-start.sh -daemon ./kafka_2.13-3.6.1/config/server.properties
# Start Redis
!apt-get install redis-server -qq > /dev/null
!service redis-server start

import time, socket, os
def wait_for_port(port, host='localhost', timeout=120):
    start_time = time.time()
    while True:
        try:
            with socket.create_connection((host, port), timeout=1):
                print(f"Service at {host}:{port} is ready!")
                return True
        except (OSError, ConnectionRefusedError):
            if time.time() - start_time > timeout:
                print(f"Timeout waiting for {host}:{port} to start.")
                # Dump logs for debugging
                if os.path.exists('minio.log'):
                    print('--- MINIO LOG ---')
                    print(open('minio.log').read())
                if os.path.exists('es.log'):
                    print('--- ES LOG ---')
                    print(open('es.log').read())
                if os.path.exists('cassandra.log'):
                    print('--- CASSANDRA LOG ---')
                    print(open('cassandra.log').read())
                raise Exception(f"Service at {host}:{port} failed to start.")
            time.sleep(2)

# Wait for services
wait_for_port(9092) # Kafka
wait_for_port(6379) # Redis


## 3. Create Kafka Topic

Creates a topic named `input-topic` with 1 partition and replication factor 1.

In [ ]:
# Create Topic
!./kafka_2.13-3.6.1/bin/kafka-topics.sh --create --topic input-topic --bootstrap-server localhost:9092 --replication-factor 1 --partitions 1

## 4. Producer

Sends User IDs to Kafka. Also pre-loads User Names into Redis.

In [ ]:
from kafka import KafkaProducer
import time, redis

# Preload Redis
print("Preloading Redis...")
r = redis.Redis()
for i in range(1, 11): r.set(f"user:{i}", f"User_{i}")

print("Starting Producer...")
producer = KafkaProducer(bootstrap_servers='localhost:9092')
for i in range(1, 10): producer.send('input-topic', f'{i}'.encode('utf-8'))
producer.flush()
print("Producer finished.")

## 5. Spark Enrichment with Redis

For each message (User ID), look up the name in Redis and print the Enriched result.

In [ ]:
%%writefile kafka_consumer.py
from pyspark.sql import SparkSession
import redis

spark = SparkSession.builder.appName("Redis").getOrCreate()

def process_batch(df, epoch_id):
    rows = df.collect()
    r_local = redis.Redis()
    for row in rows:
         uid = row.value.decode('utf-8')
         name = r_local.get(f"user:{uid}")
         if name: print(f"Enriched: {uid} -> {name.decode('utf-8')}")

df = spark.readStream.format("kafka").option("kafka.bootstrap.servers", "localhost:9092").option("subscribe", "input-topic").option("startingOffsets", "earliest").load()
query = df.selectExpr("CAST(value AS STRING)").writeStream.foreachBatch(process_batch).start()
query.awaitTermination(20)

In [ ]:
!spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 kafka_consumer.py

## 6. Verification

Check Redis keys.

In [ ]:
import redis
r = redis.Redis()
keys = r.keys("user:*")
print(f"--- Found {len(keys)} users in Redis ---")
print(keys[:5])